# 05 – İnce Ayar (Fine-Tuning)

04 numaralı not defterinde klasik yöntemlerin sınırına ulaştık: TF-IDF ve Sentence-BERT özelliklerini birleştirince en iyi Pearson 0,57, en iyi QWK 0,47 oldu. Hedef 0,70.

Sebebini de gördük: Bütün özellikler iki metin arasındaki benzerliği **dışarıdan ölçen** sayılardı; model metni kendisi okumuyordu. "Corn syrup is **denser** than water" ile "Corn syrup is **less dense** than water" gibi zıt cevaplar bu sayılarda neredeyse aynı görünüyordu.

Bu not defterinde bir dil modelini **öğretmen puanlarıyla eğiteceğiz**. Model referans cevabı ve öğrenci cevabını **birlikte okuyup** doğrudan puan tahmin etmeyi öğrenecek.

Adımlar:
- İnce ayar nedir? Çapraz kodlayıcı nedir?
- Kullanacağımız model ve eğitim ayarları
- İnce ayardan önce: Hazır model ne kadar başarılı?
- Eğitim
- Sonuçlar: Bütün yöntemlerin karşılaştırılması
- Zor örnekler: Model zıt anlamları artık ayırt edebiliyor mu?

### Nasıl çalıştırılır?
1. Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle ve çift tıklayıp Colab'da aç.
2. **GPU şart:** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet.** GPU olmadan eğitim saatler sürer; bu yüzden not defteri GPU kapalıysa durup uyarı verir.
3. **Çalışma zamanı → Tümünü çalıştır** de. Toplam süre GPU ile yaklaşık **15–20 dakika**. Eğitim sırasında sekmeyi kapatma.
4. Karşılaştırma tablosunun dolu çıkması için 04 numaralı not defterinin daha önce çalıştırılmış olması gerekir (çalıştırılmadıysa da bu not defteri çalışır, sadece tabloda önceki yöntemler görünmez).

## 1. Kurulum

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

import torch
import pandas as pd
HIZLI_TEST = os.environ.get("CEVAPANALIZ_HIZLI_TEST") == "1"   # yalnızca geliştirici testi için
CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
if CIHAZ == "cpu" and not HIZLI_TEST:
    raise RuntimeError("GPU kapalı! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seçip "
                       "not defterini baştan çalıştır. GPU olmadan eğitim saatler sürer.")
print("GPU:", torch.cuda.get_device_name(0) if CIHAZ == "cuda" else "yok (hızlı test)")

df = pd.read_csv(PROJE / "data" / "processed" / "ortak_veri.csv", keep_default_na=False)
print(f"{len(df):,} cevap yüklendi.".replace(",", "."))

## 2. İnce ayar nedir? Çapraz kodlayıcı nedir?

**Önceden eğitilmiş model:** BERT gibi dil modelleri, milyonlarca metin okuyarak dili öğrenmiştir: kelimelerin anlamlarını, cümle yapısını, "less dense" ile "denser"ın zıt olduğunu… Ama bizim işimizi (öğrenci cevabı puanlamayı) bilmez.

**İnce ayar:** Önceden eğitilmiş modeli alıp **bizim verimizle bir süre daha eğitmektir**. Dili sıfırdan öğretmeyiz; zaten bilen bir modele sadece yeni bir işi öğretiriz. Tıpkı İngilizce bilen birine "bu sınav kağıtlarını şu öğretmen gibi puanla" deyip örnek kağıtlar göstermek gibi.

**Çapraz kodlayıcı (cross-encoder) ile Sentence-BERT'in farkı:**

```
Sentence-BERT (03):   referans ──► model ──► vektör ─┐
                                                      ├──► kosinüs benzerliği
                      cevap    ──► model ──► vektör ─┘
                      (iki cümle birbirini hiç görmüyor)

Çapraz kodlayıcı (05): [referans + cevap] ──► model ──► puan
                      (model iki cümleyi aynı anda okuyor)
```

Çapraz kodlayıcı iki cümleyi **yan yana** okuduğu için referanstaki "denser" kelimesiyle cevaptaki "less dense" ifadesini doğrudan karşılaştırabilir. Üstelik öğretmen puanlarıyla eğitildiği için "bu veri setinde hangi farklar puanı düşürür?" sorusunu da öğrenir.

## 3. Kullanacağımız model ve eğitim ayarları

**Model:** `cross-encoder/stsb-distilroberta-base`
- **DistilRoBERTa:** RoBERTa dil modelinin küçültülmüş hâli (82 milyon parametre, 6 katman). GPU'da hızlı eğitilebilir.
- Bu model daha önce **cümle benzerliği** verisiyle (STS Benchmark) çapraz kodlayıcı olarak eğitilmiş: İki cümleye 0–1 arası benzerlik puanı veriyor. Yani işimize yakın bir işi zaten biliyor; biz onu öğretmen puanlarına uyarlayacağız.

**Eğitim ayarları:**

| Ayar | Değer | Anlamı |
|---|---|---|
| Epok | 3 | Model eğitim verisinin tamamını 3 kez görür |
| Yığın (batch) | 16 | Model her adımda 16 cevaba bakıp kendini günceller |
| Öğrenme oranı | 0,00002 | Her adımda ağırlıkların ne kadar değişeceği. Küçük tutulur; büyük olursa model önceden öğrendiklerini unutur |
| Isınma | adımların %10'u | Eğitimin başında öğrenme oranı yavaşça artırılır, sonra yavaşça azaltılır |
| En fazla uzunluk | 160 parça | Referans + cevap birlikte en fazla 160 kelime parçası (token) olabilir; cevaplarımız bundan kısa |

**Kayıp (loss):** Modelin tahmini ile öğretmen puanı arasındaki farkı ölçen sayı. Eğitim boyunca bu sayının **düşmesini** bekleriz; düşüyorsa model öğreniyor demektir. Aşağıda her epoktan sonra kaybı yazdıracağız.

**Test düzeni önceki not defterleriyle aynı:** Mohler'de 5 katlı çapraz doğrulama (5 ayrı model eğitilir, her biri görmediği sorularla test edilir), SciEntsBank ve Beetle'da kendi eğitim kümesiyle bir model eğitilip görülmemiş cevaplar ve görülmemiş sorularla test edilir. Toplam **7 model** eğitilecek.

In [ ]:
import time
import numpy as np
from sklearn.model_selection import GroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

MODEL_ADI = "cross-encoder/stsb-distilroberta-base"
AYAR = {"epok": 3, "batch": 16, "ogrenme_orani": 2e-5, "max_uzunluk": 160}
if HIZLI_TEST:
    MODEL_ADI, AYAR["epok"] = "cross-encoder/stsb-TinyBERT-L4", 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
GPU = CIHAZ == "cuda"

def kodla(satirlar):
    """Referans ve öğrenci cevabını modelin okuyacağı tek bir girdi hâline getirir."""
    return tokenizer(list(df.loc[satirlar, "referans_cevap"]), list(df.loc[satirlar, "ogrenci_cevap"]),
                     truncation=True, max_length=AYAR["max_uzunluk"], padding=True, return_tensors="pt").to(CIHAZ)

@torch.no_grad()
def tahmin_et(model, satirlar):
    model.eval()
    sonuc = []
    for i in range(0, len(satirlar), 128):
        with torch.autocast(device_type=CIHAZ, enabled=GPU):
            cikti = model(**kodla(satirlar[i:i + 128])).logits.squeeze(-1)
        sonuc.append(torch.sigmoid(cikti.float()).cpu().numpy())
    return np.concatenate(sonuc)

def egit(egitim_satirlari, tohum=0):
    """Önceden eğitilmiş modeli yükler ve verilen cevaplarla öğretmen puanlarını tahmin edecek şekilde eğitir."""
    torch.manual_seed(tohum)
    rastgele = np.random.default_rng(tohum)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI).to(CIHAZ)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AYAR["ogrenme_orani"], weight_decay=0.01)
    adim = AYAR["epok"] * int(np.ceil(len(egitim_satirlari) / AYAR["batch"]))
    zamanlayici = get_linear_schedule_with_warmup(optimizer, int(0.1 * adim), adim)
    olcekleyici = torch.amp.GradScaler(enabled=GPU)
    kayip_fonksiyonu = torch.nn.BCEWithLogitsLoss()   # tahmin ile 0-1 arası öğretmen puanı arasındaki fark

    for epok in range(AYAR["epok"]):
        model.train()
        sira, toplam = rastgele.permutation(egitim_satirlari), 0.0
        for i in range(0, len(sira), AYAR["batch"]):
            yigin = sira[i:i + AYAR["batch"]]
            puan = torch.tensor(df.loc[yigin, "puan"].values, dtype=torch.float32, device=CIHAZ)
            with torch.autocast(device_type=CIHAZ, enabled=GPU):
                kayip = kayip_fonksiyonu(model(**kodla(yigin)).logits.squeeze(-1).float(), puan)
            optimizer.zero_grad()
            olcekleyici.scale(kayip).backward()
            olcekleyici.step(optimizer)
            olcekleyici.update()
            zamanlayici.step()
            toplam += kayip.item() * len(yigin)
        print(f"    epok {epok + 1}/{AYAR['epok']}: ortalama kayıp {toplam / len(sira):.4f}")
    return model

print("Model:", MODEL_ADI)

## 4. İnce ayardan önce: Hazır model ne kadar başarılı?

Eğitime başlamadan önce, hazır modelin (sadece cümle benzerliği için eğitilmiş hâliyle) bizim test cevaplarımıza verdiği puanlara bakalım. Böylece eğitimin etkisini doğrudan görebileceğiz.

In [ ]:
def egitim_planlari():
    """(veri seti, eğitim satırları, {test adı: test satırları}, QWK basamağı) üretir."""
    mohler = df.index[df["veri_seti"] == "mohler"].values
    for k, (egitim, test) in enumerate(GroupKFold(n_splits=5).split(mohler, groups=df.loc[mohler, "soru_id"])):
        yield f"Mohler (kat {k + 1}/5)", mohler[egitim], {"Mohler": mohler[test]}, 5
    for veri_seti, ad in [("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
        secili = df["veri_seti"] == veri_seti
        testler = {f"{ad} – görülmemiş cevaplar": df.index[secili & (df["bolum"] == "test_ua")].values,
                   f"{ad} – görülmemiş sorular": df.index[secili & (df["bolum"] == "test_uq")].values}
        if veri_seti == "scientsbank":   # 6. bölümdeki örnekler için (sonuç tablosunda kullanılmaz)
            testler["SciEntsBank – görülmemiş alanlar"] = df.index[secili & (df["bolum"] == "test_ud")].values
        yield ad, df.index[secili & (df["bolum"] == "train")].values, testler, 2

PLANLAR = list(egitim_planlari())
if HIZLI_TEST:  # geliştirici testi: çok küçük parçalar
    PLANLAR = [(a, e[:256], {t: s[:128] for t, s in ts.items()}, q) for a, e, ts, q in PLANLAR]

QWK_SEVIYE = {t: q for _, _, ts, q in PLANLAR for t in ts}
TUM_TEST = np.unique(np.concatenate([s for _, _, ts, _ in PLANLAR for s in ts.values()]))

baslangic = time.time()
hazir_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI).to(CIHAZ)
df["once"] = np.nan
df.loc[TUM_TEST, "once"] = tahmin_et(hazir_model, TUM_TEST)
del hazir_model
print(f"{len(TUM_TEST):,} test cevabı puanlandı ({time.time() - baslangic:.0f} saniye).".replace(",", "."))

## 5. Eğitim

Şimdi 7 modeli sırayla eğitiyoruz. Her modelin her epoktan sonraki kaybı yazdırılacak; **kaybın epoktan epoğa düşmesi**, modelin öğrendiğini gösterir. Eğitilen her model, eğitimde **hiç görmediği** test cevaplarını puanlar.

Bu hücre GPU ile yaklaşık 15 dakika sürer.

In [ ]:
df["sonra"] = np.nan
toplam_baslangic = time.time()
for ad, egitim_satirlari, testler, _ in PLANLAR:
    baslangic = time.time()
    print(f"{ad}: {len(egitim_satirlari):,} cevapla eğitiliyor".replace(",", "."))
    model = egit(egitim_satirlari)
    for test_adi, test_satirlari in testler.items():
        df.loc[test_satirlari, "sonra"] = tahmin_et(model, test_satirlari)
    del model
    if GPU:
        torch.cuda.empty_cache()
    print(f"    bitti ({time.time() - baslangic:.0f} saniye)\n")
print(f"Toplam eğitim süresi: {(time.time() - toplam_baslangic) / 60:.1f} dakika")

df.loc[TUM_TEST, ["veri_seti", "bolum", "soru_id", "puan", "once", "sonra"]].to_csv(
    SONUC / "05_ince_ayar_tahminler.csv", encoding="utf-8")

## 6. Sonuçlar: Bütün yöntemlerin karşılaştırılması

İnce ayarlı modeli, önceki not defterlerinin en iyi yöntemleriyle aynı testlerde karşılaştırıyoruz:

| Yöntem | Nereden? |
|---|---|
| TF-IDF · Doğrusal | 02 – kelime tabanlı en iyi yöntem |
| TF-IDF + SBERT · Gradyan artırma | 04 – klasik yöntemlerin en iyisi |
| İnce ayardan önce | Bu not defteri – hazır çapraz kodlayıcı |
| İnce ayardan sonra | Bu not defteri – öğretmen puanlarıyla eğitilmiş çapraz kodlayıcı |

In [ ]:
from src.puanlama.olcutler import olc

TEST_SIRA = ["Mohler", "SciEntsBank – görülmemiş cevaplar", "SciEntsBank – görülmemiş sorular",
             "Beetle – görülmemiş cevaplar", "Beetle – görülmemiş sorular"]
test_satirlari = {t: np.concatenate([ts[t] for _, _, ts, _ in PLANLAR if t in ts]) for t in TEST_SIRA}

satirlar = []
for test_adi in TEST_SIRA:
    s = test_satirlari[test_adi]
    for yontem, sutun in [("İnce ayardan önce", "once"), ("İnce ayardan sonra", "sonra")]:
        satirlar.append({"test": test_adi, "yontem": yontem, **olc(df.loc[s, "puan"], df.loc[s, sutun], QWK_SEVIYE[test_adi])})
sonuc = pd.DataFrame(satirlar)
sonuc.round(4).to_csv(SONUC / "05_ince_ayar.csv", index=False, encoding="utf-8")

# Önceki not defterlerinin sonuçlarını ekle (04 çalıştırıldıysa)
onceki_dosya = SONUC / "04_ozellik_birlestirme.csv"
ONCEKI = ["TF-IDF · Doğrusal", "TF-IDF + SBERT · Gradyan artırma"]
if onceki_dosya.exists() and not HIZLI_TEST:
    onceki = pd.read_csv(onceki_dosya)
    sonuc = pd.concat([onceki[onceki["yontem"].isin(ONCEKI)], sonuc], ignore_index=True)
YONTEM_SIRA = [y for y in ONCEKI + ["İnce ayardan önce", "İnce ayardan sonra"] if y in set(sonuc["yontem"])]

for olcut in ["pearson", "qwk"]:
    print(f"\n{olcut.upper()}")
    display(sonuc.pivot(index="yontem", columns="test", values=olcut).loc[YONTEM_SIRA, TEST_SIRA].round(3))

In [ ]:
import matplotlib.pyplot as plt

RENKLER = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
RENK = {y: RENKLER[i] for i, y in enumerate(ONCEKI + ["İnce ayardan önce", "İnce ayardan sonra"])}
fig, eksenler = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
genislik = 0.8 / len(YONTEM_SIRA)
for ax, olcut, baslik in zip(eksenler, ["pearson", "qwk"], ["Pearson korelasyonu", "QWK"]):
    for i, yontem in enumerate(YONTEM_SIRA):
        degerler = sonuc[sonuc["yontem"] == yontem].set_index("test").loc[TEST_SIRA, olcut]
        x = np.arange(len(TEST_SIRA)) + (i - (len(YONTEM_SIRA) - 1) / 2) * genislik
        ax.bar(x, degerler, width=genislik - 0.03, color=RENK[yontem], label=yontem)
    ax.axhline(0.70, color="#52514e", linewidth=1, linestyle="--")
    ax.set_xticks(np.arange(len(TEST_SIRA)), [t.replace(" – ", "\n") for t in TEST_SIRA], fontsize=8.5)
    ax.set_title(baslik, loc="left", fontsize=12)
    ax.set_ylim(0, 1)
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
eksenler[0].text(len(TEST_SIRA) - 0.5, 0.71, "hedef 0,70", ha="right", va="bottom", fontsize=9, color="#52514e")
tutamaklar, etiketler = eksenler[0].get_legend_handles_labels()
fig.legend(tutamaklar, etiketler, frameon=False, loc="upper left", ncol=4, bbox_to_anchor=(0.01, 0.93), fontsize=10)
fig.suptitle("İnce ayarın etkisi", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(SEKIL / "05_ince_ayar.png", dpi=150, bbox_inches="tight")
plt.show()

### Sonuçların yorumu

Aşağıdaki hücre, tablodaki sayılara bakarak yorumu **kendisi yazar**: Her testte ince ayarın önceki en iyi yönteme göre ne kadar fark yarattığını ve hedefe ulaşılıp ulaşılmadığını söyler.

In [ ]:
def virgul(x):
    return f"{x:.2f}".replace(".", ",")

tablo = {o: sonuc.pivot(index="yontem", columns="test", values=o) for o in ["pearson", "qwk"]}
for olcut, ad in [("pearson", "Pearson"), ("qwk", "QWK")]:
    print(f"── {ad} ──")
    for test_adi in TEST_SIRA:
        sonra = tablo[olcut].loc["İnce ayardan sonra", test_adi]
        once = tablo[olcut].loc["İnce ayardan önce", test_adi]
        satir = f"{test_adi}: ince ayardan önce {virgul(once)} → sonra {virgul(sonra)}"
        onceki_en_iyi = [y for y in ONCEKI if y in tablo[olcut].index]
        if onceki_en_iyi:
            en_iyi = tablo[olcut].loc[onceki_en_iyi, test_adi].max()
            fark = sonra - en_iyi
            satir += f" | klasik yöntemlerin en iyisi {virgul(en_iyi)} ({'+' if fark >= 0 else ''}{virgul(fark)})"
        satir += " | hedefe ulaşıldı ✓" if sonra >= 0.70 else f" | hedefe {virgul(0.70 - sonra)} kaldı"
        print("  " + satir)
    print()

**Tabloyu okurken dikkat edilecekler:**
- **"İnce ayardan önce" ile "sonra" arasındaki fark**, öğretmen puanlarıyla eğitmenin etkisini gösterir.
- **"Görülmemiş cevaplar" testi**, modelin eğitimde gördüğü sorulara gelen yeni cevapları ne kadar iyi puanladığını ölçer; gerçek kullanımda öğretmenin aynı soruyu yeni öğrencilere sorduğu duruma karşılık gelir.
- **"Görülmemiş sorular" testi** daha zordur: Model o soruyu hiç görmemiştir. Bu testte başarı, sistemin yeni sorulara genellenebildiğini gösterir.

## 7. Zor örnekler: Model zıt anlamları artık ayırt edebiliyor mu?

03'te Sentence-BERT'in **zıt anlamlı** cevaplara çok yüksek benzerlik verdiğini görmüştük ("denser" / "less dense": 0,97). Aynı cevaplar SciEntsBank'in **görülmemiş alanlar** test kümesinde; yani SciEntsBank modeli bu soruyu eğitimde hiç görmedi. İnce ayardan önce ve sonra bu cevaplara verilen puanlara bakalım. 01'deki **"ortak kelimesi yok ama 5/5 almış"** Mohler cevaplarına da bakacağız.

In [ ]:
import textwrap

def yaz(baslik, metin):
    print(f"{baslik:<16}" + textwrap.fill(metin, 100, subsequent_indent=" " * 16))

def goster(satirlar, baslik):
    print("=" * 116)
    print(baslik)
    for _, satir in satirlar.iterrows():
        print("-" * 116)
        yaz("Referans:", satir["referans_cevap"])
        yaz("Öğrenci:", satir["ogrenci_cevap"])
        print(f"{'Öğretmen:':<16}{virgul(satir['puan'])}   ince ayardan önce: {virgul(satir['once'])}   "
              f"sonra: {virgul(satir['sonra'])}  (0-1 ölçeğinde)")

misir = df[df["ogrenci_cevap"].str.contains("corn syrup", case=False) & (df["orijinal_puan"] == "contradictory")
           & df["sonra"].notna()].head(3)
if len(misir):
    goster(misir, "Zıt anlamlı (çelişkili) cevaplar – öğretmen puanı 0")

from src.puanlama.ozellikler import kelimeler, ngram_ortusme
mohler = df[(df["veri_seti"] == "mohler") & df["sonra"].notna() & (df["puan"] == 1.0)].copy()
mohler["ortusme"] = [ngram_ortusme(r, c) for r, c in zip(mohler["referans_cevap"], mohler["ogrenci_cevap"])]
mohler = mohler[mohler["ogrenci_cevap"].map(lambda s: len(kelimeler(s))) >= 6]
goster(mohler.nsmallest(3, "ortusme"), "Ortak kelimesi olmayan ama tam puan almış cevaplar – öğretmen puanı 1")

In [ ]:
etiket_tr = {"correct": "doğru", "partially_correct_incomplete": "kısmen doğru", "contradictory": "çelişkili",
             "irrelevant": "alakasız", "non_domain": "alan dışı"}
semeval_test = np.concatenate([test_satirlari[t] for t in TEST_SIRA if t != "Mohler"])
ozet = df.loc[semeval_test].groupby("orijinal_puan")[["once", "sonra"]].mean().rename(index=etiket_tr)
ozet.columns = ["ince ayardan önce (ortalama tahmin)", "ince ayardan sonra (ortalama tahmin)"]
print("SciEntsBank ve Beetle test cevaplarında, öğretmen etiketine göre modelin verdiği ortalama puan (0-1):")
display(ozet.loc[[e for e in etiket_tr.values() if e in ozet.index]].round(3))

**Bu tabloyu nasıl okuyalım?** İyi bir model "doğru" cevaplara yüksek, "kısmen doğru" cevaplara orta, "çelişkili" ve "alakasız" cevaplara düşük puan vermelidir. 03'te Sentence-BERT'te çelişkili cevapların benzerliği (0,51) kısmen doğru cevaplarınkine (0,53) neredeyse eşitti. İnce ayardan sonra **çelişkili cevapların ortalama puanı kısmen doğru cevaplarınkinden belirgin şekilde aşağı indiyse**, model zıtlığı ayırt etmeyi öğrenmiş demektir.

## Özet

Bu not defterinde:
- **İnce ayar** kavramını ve iki cümleyi birlikte okuyan **çapraz kodlayıcıyı** tanıdık.
- Hazır bir çapraz kodlayıcıyı (DistilRoBERTa) **öğretmen puanlarıyla eğittik** ve 7 model eğittik.
- İnce ayarlı modeli önceki bütün yöntemlerle aynı testlerde karşılaştırdık; sonuçların yorumu yukarıda.
- Zıt anlamlı cevaplar ve "ortak kelimesi yok ama doğru" cevaplar gibi **zor örneklerde** modelin davranışına baktık.

Sonuçlar Drive'da:
- `CevapAnaliz/sonuclar/05_ince_ayar.csv` – karşılaştırma tablosu
- `CevapAnaliz/sonuclar/05_ince_ayar_tahminler.csv` – her test cevabı için ince ayardan önce ve sonraki tahminler
- `CevapAnaliz/sekiller/05_ince_ayar.png` – karşılaştırma grafiği

**Bu sonuçlar 4. hafta raporuna girecek.** Karşılaştırma tablosunun ve yorum hücresinin çıktısının ekran görüntüsünü paylaşırsan rapora birlikte yazarız.